# NumPy Interview Refresher

Build fast recall for shapes, combining, broadcasting, indexing, vectorization, ranking, linear algebra, and ML primitives.

- **Study time:** 40-50 minutes
- **Prerequisites:** basic Python expressions and loops
- **Mode:** `quick`
- **Data policy:** no external files or downloads; seeded synthetic arrays only
- **Provenance:** consolidated from the legacy NumPy refresher variants and advanced curated cells

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(42)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


show("Environment | NumPy version", np.__version__)


--- Environment | NumPy version ---
2.5.2


## 1. Creation and dtype

Predict each shape and dtype before running the cell. An ML implementation can silently fail when an integer array truncates a floating-point update.


In [2]:
vector = np.array([1, 2, 3])
matrix = np.array([[1, 2], [3, 4]], dtype=np.float32)
float_vector = vector.astype(np.float64)
zeros = np.zeros((2, 3))
identity = np.eye(3)
samples = rng.normal(size=(3, 4))

show("Creation | vector (value, dtype, shape)", (vector, vector.dtype, vector.shape))
show(
    "Creation | matrix metadata",
    {"shape": matrix.shape, "ndim": matrix.ndim, "size": matrix.size, "dtype": matrix.dtype},
)
show("Casting | integer vector to floating dtype", float_vector.dtype)
show("Creation | zeros (2 x 3)", zeros)
show("Creation | identity matrix", identity)
show("Creation | seeded normal samples", samples)


--- Creation | vector (value, dtype, shape) ---
(array([1, 2, 3]), dtype('int64'), (3,))

--- Creation | matrix metadata ---
{'shape': (2, 2), 'ndim': 2, 'size': 4, 'dtype': dtype('float32')}

--- Casting | integer vector to floating dtype ---
float64

--- Creation | zeros (2 x 3) ---
[[0. 0. 0.]
 [0. 0. 0.]]

--- Creation | identity matrix ---
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

--- Creation | seeded normal samples ---
[[ 0.305 -1.04   0.75   0.941]
 [-1.951 -1.302  0.128 -0.316]
 [-0.017 -0.853  0.879  0.778]]


## 2. Shape changes, views, and copies


In [3]:
base = np.arange(12)
reshaped = base.reshape(3, 4)
raveled = reshaped.ravel()
flattened = reshaped.flatten()
transposed = reshaped.T
expanded_front = reshaped[None, :, :]  # (1, 3, 4)
expanded_last = np.expand_dims(reshaped, -1)  # (3, 4, 1)
squeezed = np.squeeze(expanded_last, axis=-1)  # (3, 4)

show("Shape | base -> reshaped", f"{base.shape} -> {reshaped.shape}")
show("Memory | ravel shares memory", np.shares_memory(reshaped, raveled))
show("Memory | flatten shares memory", np.shares_memory(reshaped, flattened))
show("Shape/memory | transpose", (transposed.shape, np.shares_memory(reshaped, transposed)))
show(
    "Shape | add/remove singleton axes",
    (expanded_front.shape, expanded_last.shape, squeezed.shape),
)


--- Shape | base -> reshaped ---
(12,) -> (3, 4)

--- Memory | ravel shares memory ---
True

--- Memory | flatten shares memory ---
False

--- Shape/memory | transpose ---
((4, 3), True)

--- Shape | add/remove singleton axes ---
((1, 3, 4), (3, 4, 1), (3, 4))


## 3. Combining and splitting arrays

`concatenate` joins along an existing axis, so every other dimension must match. `stack` inserts a new axis, so every input shape must match. `vstack` and `hstack` are conveniences; prefer an explicit axis when 1-D behavior could be ambiguous.


In [4]:
left = np.arange(6).reshape(2, 3)
right = left + 10

concatenated_rows = np.concatenate([left, right], axis=0)  # (4, 3)
concatenated_columns = np.concatenate([left, right], axis=1)  # (2, 6)
stacked_front = np.stack([left, right], axis=0)  # (2, 2, 3)
stacked_last = np.stack([left, right], axis=-1)  # (2, 3, 2)
equal_halves = np.split(concatenated_rows, 2, axis=0)
uneven_chunks = np.array_split(np.arange(7), 3)

show("Combine | input shapes", (left.shape, right.shape))
show(
    "Combine | concatenate along existing axes",
    {"axis=0": concatenated_rows.shape, "axis=1": concatenated_columns.shape},
)
show(
    "Combine | stack along new axes",
    {"axis=0": stacked_front.shape, "axis=-1": stacked_last.shape},
)
show(
    "Convenience | vstack/hstack equal explicit concatenate",
    (
        np.array_equal(np.vstack([left, right]), concatenated_rows),
        np.array_equal(np.hstack([left, right]), concatenated_columns),
    ),
)
show(
    "Split | equal and uneven chunk shapes",
    ([part.shape for part in equal_halves], [part.shape for part in uneven_chunks]),
)


--- Combine | input shapes ---
((2, 3), (2, 3))

--- Combine | concatenate along existing axes ---
{'axis=0': (4, 3), 'axis=1': (2, 6)}

--- Combine | stack along new axes ---
{'axis=0': (2, 2, 3), 'axis=-1': (2, 3, 2)}

--- Convenience | vstack/hstack equal explicit concatenate ---
(True, True)

--- Split | equal and uneven chunk shapes ---
([(2, 3), (2, 3)], [(3,), (2,), (2,)])


## 4. Slicing, boolean masks, and fancy indexing


In [5]:
A = np.arange(20).reshape(4, 5)
sliced = A[:2, 1:4]
mask = A % 3 == 0
selected = A[mask]
rows = np.array([0, 2, 3])
columns = np.array([1, 4, 0])
paired = A[rows, columns]
binary = np.where(A > 12, 1, 0)
where_rows, where_columns = np.where(A > 12)
where_coordinates = np.column_stack([where_rows, where_columns])

show("Indexing | source A", A)
show("Indexing | basic slice A[:2, 1:4]", sliced)
show("Memory | slice shares memory with A", np.shares_memory(A, sliced))
show("Indexing | boolean mask dtype and shape", (mask.dtype, mask.shape))
show("Indexing | values divisible by three", selected)
show("Indexing | paired fancy selection A[rows, columns]", paired)
show("Selection | np.where(A > 12, 1, 0)", binary)
show("Selection | coordinates returned by np.where(condition)", where_coordinates)


--- Indexing | source A ---
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]

--- Indexing | basic slice A[:2, 1:4] ---
[[1 2 3]
 [6 7 8]]

--- Memory | slice shares memory with A ---
True

--- Indexing | boolean mask dtype and shape ---
(dtype('bool'), (4, 5))

--- Indexing | values divisible by three ---
[ 0  3  6  9 12 15 18]

--- Indexing | paired fancy selection A[rows, columns] ---
[ 1 14 15]

--- Selection | np.where(A > 12, 1, 0) ---
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 1]
 [1 1 1 1 1]]

--- Selection | coordinates returned by np.where(condition) ---
[[2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [3 4]]


## 5. Broadcasting

Align shapes from the right. Adding singleton axes turns pairwise operations into ordinary elementwise arithmetic.


In [6]:
X = rng.normal(size=(5, 2))  # (n=5, d=2)
centers = rng.normal(size=(3, 2))  # (k=3, d=2)
mean = X.mean(axis=0, keepdims=True)
centered = X - mean
differences = X[:, None, :] - centers[None, :, :]  # (5, 3, 2)
squared_distances = np.sum(differences**2, axis=2)  # (5, 3)

show("Broadcast | X, mean, centered shapes", (X.shape, mean.shape, centered.shape))
show("Broadcast | pairwise difference shape", differences.shape)
show("Broadcast | pairwise squared distances (n x k)", squared_distances)


--- Broadcast | X, mean, centered shapes ---
((5, 2), (1, 2), (5, 2))

--- Broadcast | pairwise difference shape ---
(5, 3, 2)

--- Broadcast | pairwise squared distances (n x k) ---
[[2.98  2.433 0.798]
 [1.067 1.06  1.504]
 [1.376 1.003 1.781]
 [0.129 1.799 0.292]
 [2.258 0.167 1.609]]


## 6. Reductions and standardization


In [7]:
X = rng.normal(loc=100, scale=10, size=(6, 3))
feature_mean = X.mean(axis=0, keepdims=True)
feature_std = X.std(axis=0, keepdims=True)
standardized = (X - feature_mean) / (feature_std + 1e-12)

show("Reduction | X.sum(axis=0) shape", X.sum(axis=0).shape)
show("Reduction | X.sum(axis=1) shape", X.sum(axis=1).shape)
show("Standardization | feature means after transform", standardized.mean(axis=0))
show("Standardization | feature std after transform", standardized.std(axis=0))


--- Reduction | X.sum(axis=0) shape ---
(3,)

--- Reduction | X.sum(axis=1) shape ---
(6,)

--- Standardization | feature means after transform ---
[ 0. -0. -0.]

--- Standardization | feature std after transform ---
[1. 1. 1.]


## 7. Sorting, top-k, and aligned gather


In [8]:
scores = rng.normal(size=(3, 8))
k = 3
candidate_indices = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
candidate_scores = np.take_along_axis(scores, candidate_indices, axis=1)
candidate_order = np.argsort(-candidate_scores, axis=1)
topk_indices = np.take_along_axis(candidate_indices, candidate_order, axis=1)
topk_scores = np.take_along_axis(scores, topk_indices, axis=1)

show("Top-k | source scores", scores)
show("Top-k | sorted indices per row", topk_indices)
show("Top-k | aligned sorted scores", topk_scores)


--- Top-k | source scores ---
[[ 0.871  0.224  0.679  0.068  0.289  0.631 -1.457 -0.32 ]
 [-0.47  -0.639 -0.275  1.495 -0.866  0.968 -1.683 -0.335]
 [ 0.163  0.586  0.711  0.793 -0.349 -0.462  0.858 -0.191]]

--- Top-k | sorted indices per row ---
[[0 2 5]
 [3 5 2]
 [6 3 2]]

--- Top-k | aligned sorted scores ---
[[ 0.871  0.679  0.631]
 [ 1.495  0.968 -0.275]
 [ 0.858  0.793  0.711]]


## 8. Linear algebra: solve, norms, and SVD


In [9]:
A = rng.normal(size=(4, 4))
b = rng.normal(size=4)
solution = np.linalg.solve(A, b)
residual = A @ solution - b
U, singular_values, Vt = np.linalg.svd(A, full_matrices=False)

show("Linear algebra | solve residual max abs", np.max(np.abs(residual)))
show("Linear algebra | row L2 norms", np.linalg.norm(A, axis=1))
show("Linear algebra | compact SVD shapes", (U.shape, singular_values.shape, Vt.shape))


--- Linear algebra | solve residual max abs ---
2.220446049250313e-16

--- Linear algebra | row L2 norms ---
[2.001 0.839 1.065 1.395]

--- Linear algebra | compact SVD shapes ---
((4, 4), (4,), (4, 4))


## 9. Stable softmax and cosine similarity


In [10]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


logits = np.array([[1000.0, 1001.0, 999.0], [1.0, 0.0, -1.0]])
probabilities = softmax(logits)

left = rng.normal(size=(4, 3))
right = rng.normal(size=(5, 3))
left_unit = left / (np.linalg.norm(left, axis=1, keepdims=True) + 1e-12)
right_unit = right / (np.linalg.norm(right, axis=1, keepdims=True) + 1e-12)
cosine = left_unit @ right_unit.T

show("Softmax | probabilities", probabilities)
show("Softmax | row sums", probabilities.sum(axis=1))
show("Cosine similarity | output shape", cosine.shape)
show("Cosine similarity | first 2 x 3 block", cosine[:2, :3])


--- Softmax | probabilities ---
[[0.245 0.665 0.09 ]
 [0.665 0.245 0.09 ]]

--- Softmax | row sums ---
[1. 1.]

--- Cosine similarity | output shape ---
(4, 5)

--- Cosine similarity | first 2 x 3 block ---
[[0.087 0.123 0.348]
 [0.049 0.922 0.507]]


## 10. Scatter-add, sliding windows, and NaN-aware reduction


In [11]:
repeated_indices = np.array([0, 1, 1, 3, 3, 3])
values = np.array([10, 1, 1, 5, 2, 2])
accumulated = np.zeros(5, dtype=int)
np.add.at(accumulated, repeated_indices, values)

from numpy.lib.stride_tricks import sliding_window_view

sequence = np.arange(10)
windows = sliding_window_view(sequence, window_shape=4)
values_with_nan = np.array([1.0, np.nan, 3.0, np.nan, 5.0])

show("Scatter-add | accumulated repeated indices", accumulated)
show("Sliding window | shape", windows.shape)
show("Sliding window | moving averages", windows.mean(axis=1))
show(
    "NaN-aware reduction | mean vs nanmean", (np.mean(values_with_nan), np.nanmean(values_with_nan))
)


--- Scatter-add | accumulated repeated indices ---
[10  2  0  9  0]

--- Sliding window | shape ---
(7, 4)

--- Sliding window | moving averages ---
[1.5 2.5 3.5 4.5 5.5 6.5 7.5]

--- NaN-aware reduction | mean vs nanmean ---
(np.float64(nan), np.float64(3.0))


## 11. Retrieval drills

Re-type these from a blank cell later: concatenate versus stack, standardization, pairwise distances, stable softmax, and vectorized binary metrics.


In [12]:
y_true = rng.integers(0, 2, size=100)
y_pred = rng.integers(0, 2, size=100)
true_positive = np.sum((y_true == 1) & (y_pred == 1))
false_positive = np.sum((y_true == 0) & (y_pred == 1))
false_negative = np.sum((y_true == 1) & (y_pred == 0))
precision = true_positive / (true_positive + false_positive + 1e-12)
recall = true_positive / (true_positive + false_negative + 1e-12)
f1 = 2 * precision * recall / (precision + recall + 1e-12)

assert np.allclose(probabilities.sum(axis=1), 1.0)
assert topk_indices.shape == (3, 3)
assert squared_distances.shape == (5, 3)
assert concatenated_rows.shape == (4, 3)
assert stacked_front.shape == (2, 2, 3)
assert sum(part.size for part in uneven_chunks) == 7
show("Binary metrics | precision, recall, F1", np.round([precision, recall, f1], 3))
show("Drill checks | status", "all assertions passed")


--- Binary metrics | precision, recall, F1 ---
[0.542 0.578 0.559]

--- Drill checks | status ---
all assertions passed
